# Перенос и смешивание двух стилей — версия 2.0

    **Цель:** Понять content loss, матрицу Грама и оптимизацию изображения.

    **Алгоритм:** Признаки VGG19 фиксированы; обучается изображение. Для каждого слоя отдельно рассчитываются оба style loss.

    Учебная модернизация 2026-10-01. Код адаптирован с Python 3.7 на Python 3.12. Исходные алгоритмы сохранены; историческая
    версия доступна в Git на коммите `a842320`. Исправления учебных ошибок описаны
    в README. Это учебное пособие, не промышленная система и не benchmark.

    По умолчанию: CPU, короткий запуск, синтетические демонстрационные данные,
    без загрузок из сети. Реальные данные включаются отдельно: `MLLAB_DATA_SOURCE`.
    Синтетические результаты нельзя выдавать за качество на реальных данных.
    После изменения настроек перезапустите ядро и выполните ячейки сверху вниз.
    Окружение и структура данных описаны в `docs/ENVIRONMENTS.md` и `docs/DATA.md`.


In [ ]:
from pathlib import Path
import os, sys
start = Path(os.environ.get('MLLAB_ROOT', str(Path.cwd()))).resolve()
root = next((p for p in (start, *start.parents) if (p / 'ml_lab' / 'runtime.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Откройте папку репозитория или задайте MLLAB_ROOT.')
sys.path.insert(0, str(root))
from ml_lab.runtime import settings
cfg = settings('neural-style-transfer-pytorch')


## Изображения и VGG19
Demo без сети использует случайные веса: это проверка механики, не художественного качества.


In [ ]:
from ml_lab.runtime import torch_setup
torch, device = torch_setup(cfg)
from torchvision import models, transforms
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
if cfg.source == 'demo':
    rng = np.random.default_rng(cfg.seed)
    pictures = [Image.fromarray(rng.integers(0, 256, (32, 48, 3), dtype=np.uint8)) for _ in range(3)]
    weights = None
elif cfg.source == 'local':
    pictures = []
    for name in ['content.jpg', 'style1.jpg', 'style2.jpg']:
        with Image.open(cfg.data / 'style' / name) as image:
            pictures.append(image.convert('RGB').copy())
    weights = models.VGG19_Weights.DEFAULT
    filename = weights.url.rsplit('/', 1)[1]
    cached = Path(torch.hub.get_dir()) / 'checkpoints' / filename
    if not cached.exists() and not cfg.allow_download:
        raise FileNotFoundError('VGG19 weights не закэшированы. Явно разрешите MLLAB_ALLOW_DOWNLOAD=1.')
else:
    raise ValueError('Выберите demo или local')
size = (48, 32) if cfg.quick else (350, 250)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((.485, .456, .406), (.229, .224, .225))])
content, style1, style2 = [transform(image.resize(size, Image.Resampling.LANCZOS)).unsqueeze(0).to(device) for image in pictures]
feature_net = models.vgg19(weights=weights).features[:15].to(device).eval().requires_grad_(False)
selected = {0, 2, 5, 7, 9, 11, 14}
def features(image):
    results = []
    for i, layer in enumerate(feature_net):
        image = layer(image)
        if i in selected: results.append(image)
    return results
def gram(feature):
    _, channels, height, width = feature.shape
    flattened = feature.reshape(channels, height * width)
    return flattened @ flattened.T / (channels * height * width)


## Оптимизация
Списки признаков не конкатенируются: оба стиля участвуют на каждом слое. Веса VGG не обучаются.


In [ ]:
with torch.no_grad():
    content_features = features(content)
    grams1 = [gram(f) for f in features(style1)]
    grams2 = [gram(f) for f in features(style2)]
target = content.clone().requires_grad_(True)
optimizer = torch.optim.Adam([target], lr=.03)
alpha, beta = .5, .5
for step in range(2 if cfg.quick else 600):
    optimizer.zero_grad()
    target_features = features(target)
    content_loss = sum(torch.nn.functional.mse_loss(a, b) for a, b in zip(target_features, content_features))
    style_loss1 = sum(torch.nn.functional.mse_loss(gram(a), b) for a, b in zip(target_features, grams1))
    style_loss2 = sum(torch.nn.functional.mse_loss(gram(a), b) for a, b in zip(target_features, grams2))
    loss = content_loss + 100 * (alpha * style_loss1 + beta * style_loss2)
    loss.backward(); optimizer.step()
    assert all(p.grad is None for p in feature_net.parameters())
    print(step, 'content:', content_loss.item(), 'style1:', style_loss1.item(), 'style2:', style_loss2.item())
mean = torch.tensor([.485, .456, .406], device=device).reshape(1, 3, 1, 1)
std = torch.tensor([.229, .224, .225], device=device).reshape(1, 3, 1, 1)
result = (target.detach() * std + mean).clamp(0, 1)[0].cpu().permute(1, 2, 0).numpy()
plt.imshow(result); plt.axis('off'); plt.show()
Image.fromarray((result * 255).astype('uint8')).save(cfg.output / 'mixed-style.png')


## Результат и самостоятельная работа

    Выполненные ячейки показывают формы данных, обучение и результат алгоритма.
    Успешный короткий запуск проверяет работоспособность, а не достижение заданной точности.

    1. Измените веса двух стилей.
2. Проверьте, что оба style loss влияют на итог.
3. Объясните нормализацию матрицы Грама.
4. Сравните замороженный случайный VGG с предобученным: только последний пригоден для полноценного переноса.

    Перед выводом о качестве используйте отдельные train/validation/test,
    документируйте источник данных и сравнивайте эксперименты при одинаковых настройках.
